# Overview — Contraceptive Method Choice Dataset

**Source:** UCI Machine Learning Repository — ID 30  
**Objective:** Understand the data before injecting artificial missingness.

This dataset is a subset of the 1987 National Indonesia Contraceptive Prevalence Survey.  
The samples are married women who were either not pregnant or do not know if they were pregnant at the time of interview.

The target variable is **contraceptive_method** — current contraceptive method used:
- **1** = No-use  
- **2** = Long-term methods (IUD, sterilization)  
- **3** = Short-term methods (pill, condom, injection, etc.)

**Business question:** Can we predict a woman's contraceptive choice from her socio-demographic and economic background?

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ucimlrepo import fetch_ucirepo

## 1. Load the Dataset

In [ ]:
cmc = fetch_ucirepo(id=30)
df = pd.concat([cmc.data.features.copy(),
                cmc.data.targets.rename(columns={cmc.data.targets.columns[0]: "contraceptive_method"})],
               axis=1)

print(f"Dimensions: {df.shape[0]} observations × {df.shape[1]} variables")
df.head(10)

## 2. Variable Types and Missing Values

In [ ]:
info = pd.DataFrame({
    "dtype":     df.dtypes,
    "n_unique":  df.nunique(),
    "missing":   df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(2),
    "min":       df.min(),
    "max":       df.max(),
    "example":   df.iloc[0]
})
info

### Variable Codebook

| Variable | Type | Description |
|---|---|---|
| `wife_age` | Numeric | Wife's age (years) |
| `wife_education` | Ordinal (1–4) | 1=low, 4=high |
| `husband_education` | Ordinal (1–4) | 1=low, 4=high |
| `num_children` | Numeric | Number of children ever born |
| `wife_religion` | Binary | 0=Non-Islam, 1=Islam |
| `wife_working` | Binary | 0=Yes (working), 1=No (not working) |
| `husband_occupation` | Ordinal (1–4) | Occupation category |
| `standard_of_living` | Ordinal (1–4) | 1=low, 4=high |
| `media_exposure` | Binary | 0=Good exposure, 1=Not good |
| `contraceptive_method` | Target (1/2/3) | 1=No-use, 2=Long-term, 3=Short-term |

In [ ]:
ordinal_cols = ["wife_education", "husband_education", "husband_occupation", "standard_of_living"]
binary_cols  = ["wife_religion", "wife_working", "media_exposure"]
numeric_cols = ["wife_age", "num_children"]
target_col   = "contraceptive_method"

print(f"Ordinal variables ({len(ordinal_cols)}): {ordinal_cols}")
print(f"Binary variables  ({len(binary_cols)}): {binary_cols}")
print(f"Numeric variables ({len(numeric_cols)}): {numeric_cols}")
print(f"Target: {target_col}")

## 3. Descriptive Statistics

In [ ]:
df.describe().T.round(2)

## 4. Target Variable — Contraceptive Method

In [ ]:
label_map = {1: "1 – No-use", 2: "2 – Long-term", 3: "3 – Short-term"}
df["method_label"] = df[target_col].map(label_map)

vc = df["method_label"].value_counts().sort_index()
pct = (vc / len(df) * 100).round(1)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Count", "Percentage"],
                    specs=[[{"type": "bar"}, {"type": "pie"}]])

fig.add_trace(go.Bar(x=vc.index, y=vc.values,
                     marker_color=["steelblue", "coral", "mediumseagreen"],
                     text=vc.values, textposition="outside",
                     showlegend=False), row=1, col=1)

fig.add_trace(go.Pie(labels=vc.index, values=vc.values,
                     marker_colors=["steelblue", "coral", "mediumseagreen"],
                     hole=0.3), row=1, col=2)

fig.update_layout(title_text="Distribution of Contraceptive Method (target)",
                  height=400)
fig.show()

for label, count, p in zip(vc.index, vc.values, pct.values):
    print(f"  {label:<22} {count:>4}  ({p}%)")

## 5. Distributions of Continuous Variables

In [ ]:
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=["Distribution of wife_age", "Boxplot of wife_age",
                                    "Distribution of num_children", "Boxplot of num_children"])

fig.add_trace(go.Histogram(x=df["wife_age"], nbinsx=25,
                           marker_color="steelblue", showlegend=False), row=1, col=1)
fig.add_trace(go.Box(y=df["wife_age"], marker_color="steelblue",
                     boxmean=True, showlegend=False), row=1, col=2)
fig.add_trace(go.Histogram(x=df["num_children"], nbinsx=20,
                           marker_color="coral", showlegend=False), row=2, col=1)
fig.add_trace(go.Box(y=df["num_children"], marker_color="coral",
                     boxmean=True, showlegend=False), row=2, col=2)

fig.update_layout(title_text="Continuous Variables", height=600)
fig.show()

print(df[["wife_age", "num_children"]].describe().T.round(2))

## 6. Distributions of Ordinal and Binary Variables

In [ ]:
all_cat = ordinal_cols + binary_cols
n = len(all_cat)
cols_per_row = 3
n_rows = int(np.ceil(n / cols_per_row))

fig = make_subplots(rows=n_rows, cols=cols_per_row,
                    subplot_titles=all_cat)

for i, col in enumerate(all_cat):
    r = i // cols_per_row + 1
    c = i % cols_per_row + 1
    vc = df[col].value_counts().sort_index()
    fig.add_trace(go.Bar(x=vc.index.astype(str), y=vc.values,
                         marker_color="mediumpurple", showlegend=False), row=r, col=c)

fig.update_layout(title_text="Ordinal and Binary Variables — Frequency",
                  height=350 * n_rows)
fig.show()

## 7. Feature–Target Relationships

### 7.1 Continuous Variables vs Contraceptive Method

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["wife_age by method", "num_children by method"])

colors = {"1 – No-use": "steelblue",
          "2 – Long-term": "coral",
          "3 – Short-term": "mediumseagreen"}

for label, color in colors.items():
    sub = df[df["method_label"] == label]
    fig.add_trace(go.Box(y=sub["wife_age"], name=label,
                         marker_color=color, boxmean=True), row=1, col=1)
    fig.add_trace(go.Box(y=sub["num_children"], name=label,
                         marker_color=color, boxmean=True, showlegend=False), row=1, col=2)

fig.update_yaxes(title_text="wife_age", row=1, col=1)
fig.update_yaxes(title_text="num_children", row=1, col=2)
fig.update_layout(title_text="Continuous Features by Contraceptive Method",
                  height=450)
fig.show()

for col in ["wife_age", "num_children"]:
    print(f"\n{col} by contraceptive_method:")
    print(df.groupby("method_label")[col].agg(["mean", "median"]).round(2).to_string())

### 7.2 Ordinal and Binary Variables vs Contraceptive Method

In [ ]:
fig = make_subplots(rows=3, cols=3,
                    subplot_titles=all_cat)

for i, col in enumerate(all_cat):
    r = i // 3 + 1
    c = i % 3 + 1
    ct = pd.crosstab(df[col], df[target_col], normalize="index") * 100
    for m, color in zip([1, 2, 3], ["steelblue", "coral", "mediumseagreen"]):
        if m in ct.columns:
            fig.add_trace(go.Bar(
                x=ct.index.astype(str), y=ct[m].round(1),
                name=label_map[m],
                marker_color=color,
                showlegend=(i == 0)
            ), row=r, col=c)

fig.update_layout(title_text="Contraceptive Method Distribution (%) by Feature Value",
                  barmode="stack",
                  height=900,
                  legend_title_text="Method")
fig.show()

### 7.3 Age × Number of Children — colored by method

In [ ]:
fig = px.scatter(df, x="wife_age", y="num_children",
                 color="method_label",
                 color_discrete_map=colors,
                 opacity=0.6,
                 title="Wife Age vs Number of Children — colored by Contraceptive Method",
                 labels={"wife_age": "Wife Age",
                         "num_children": "Number of Children",
                         "method_label": "Method"})
fig.show()

### 7.4 Education Level vs Contraceptive Use

In [ ]:
edu_labels = {1: "1-Low", 2: "2", 3: "3", 4: "4-High"}

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Wife Education vs Method",
                                    "Husband Education vs Method"])

for col, c in [("wife_education", 1), ("husband_education", 2)]:
    ct = pd.crosstab(df[col], df[target_col], normalize="index") * 100
    ct.index = ct.index.map(edu_labels)
    for m, color in zip([1, 2, 3], ["steelblue", "coral", "mediumseagreen"]):
        if m in ct.columns:
            fig.add_trace(go.Bar(
                x=ct.index.astype(str), y=ct[m].round(1),
                name=label_map[m],
                marker_color=color,
                showlegend=(c == 1)
            ), row=1, col=c)

fig.update_layout(title_text="Education Level vs Contraceptive Method (%)",
                  barmode="stack", height=450, legend_title_text="Method")
fig.update_yaxes(title_text="% within education level")
fig.show()

## 8. Correlation Matrix

In [ ]:
# Drop label column for correlation
num_df = df.drop(columns=["method_label"])
corr = num_df.corr().round(2)

fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=corr.columns,
    y=corr.index,
    colorscale="RdBu",
    zmid=0,
    text=corr.values,
    texttemplate="%{text}",
    colorbar=dict(title="r")
))
fig.update_layout(title="Pearson Correlation Matrix",
                  width=700, height=650)
fig.show()

# Strong pairs
corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack().reset_index()
)
corr_pairs.columns = ["var1", "var2", "r"]
strong = corr_pairs[corr_pairs["r"].abs() > 0.25].sort_values("r", ascending=False)
print("Pairs with |r| > 0.25:")
print(strong.to_string(index=False))

## 9. Correlation with Target

In [ ]:
feature_cols = [c for c in num_df.columns if c != target_col]
corr_with_target = num_df[feature_cols + [target_col]].corr()[target_col].drop(target_col).sort_values()

fig = px.bar(x=corr_with_target.values, y=corr_with_target.index,
             orientation="h",
             color=corr_with_target.values,
             color_continuous_scale="RdBu",
             color_continuous_midpoint=0,
             title="Pearson Correlation with contraceptive_method",
             labels={"x": "Correlation", "y": "Variable"})
fig.update_layout(coloraxis_showscale=False)
fig.show()

print("Top 3 positive correlations with target:")
print(corr_with_target.tail(3).to_string())
print("\nTop 3 negative correlations with target:")
print(corr_with_target.head(3).to_string())

## 10. Standard of Living and Media Exposure

In [ ]:
sol_labels = {1: "1-Low", 2: "2", 3: "3", 4: "4-High"}

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Standard of Living vs Method",
                                    "Media Exposure vs Method"])

for col, label_dict, c in [
    ("standard_of_living", sol_labels, 1),
    ("media_exposure",     {0: "Good", 1: "Not good"}, 2)
]:
    ct = pd.crosstab(df[col], df[target_col], normalize="index") * 100
    ct.index = ct.index.map(label_dict)
    for m, color in zip([1, 2, 3], ["steelblue", "coral", "mediumseagreen"]):
        if m in ct.columns:
            fig.add_trace(go.Bar(
                x=ct.index.astype(str), y=ct[m].round(1),
                name=label_map[m], marker_color=color,
                showlegend=(c == 1)
            ), row=1, col=c)

fig.update_layout(title_text="Socioeconomic Context vs Contraceptive Method",
                  barmode="stack", height=450, legend_title_text="Method")
fig.update_yaxes(title_text="% within group")
fig.show()

## 11. Religion and Work Status

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Religion vs Method",
                                    "Wife Working vs Method"])

for col, label_dict, c in [
    ("wife_religion", {0: "Non-Islam", 1: "Islam"}, 1),
    ("wife_working",  {0: "Working", 1: "Not working"}, 2)
]:
    ct = pd.crosstab(df[col], df[target_col], normalize="index") * 100
    ct.index = ct.index.map(label_dict)
    for m, color in zip([1, 2, 3], ["steelblue", "coral", "mediumseagreen"]):
        if m in ct.columns:
            fig.add_trace(go.Bar(
                x=ct.index.astype(str), y=ct[m].round(1),
                name=label_map[m], marker_color=color,
                showlegend=(c == 1)
            ), row=1, col=c)

fig.update_layout(title_text="Religion and Work Status vs Contraceptive Method",
                  barmode="stack", height=450, legend_title_text="Method")
fig.update_yaxes(title_text="% within group")
fig.show()

## 12. Quick Conclusions

| Finding | Detail |
|---|---|
| **Class imbalance** | No-use (42.7%) > Short-term (22.6%) ≈ Long-term (22.9%). Mild imbalance — worth noting for modelling |
| **Age** | Long-term method users tend to be older (more children, family complete). Short-term users are younger. No-use is bimodal |
| **Number of children** | Strong predictor — higher parity associated with long-term or no-use (sterilization / completed family) |
| **Education** | Higher education (wife and husband) shifts preference toward short-term and away from no-use |
| **Standard of living** | Higher standard of living correlates with greater contraceptive use (both types) |
| **Religion** | Islamic women show higher no-use rates, consistent with religious constraints on contraception |
| **Media exposure** | Women with good media exposure show slightly higher contraceptive use |
| **Correlations** | `num_children`–`wife_age` strong positive; education levels correlated with each other and with standard of living |
| **No missing values** | Dataset is complete — ideal starting point for artificial missingness injection |